# JN4 - Dated milestone events + the completion stage

**Curriculum notebook 4 of 6.** A building moves through milestones: a permit is *issued*, then the building is *finaled* (occupiable). This notebook turns the structured date columns into **dated events**, then **derives** each building's stage from them - and never from a status string.

> Clonable + **read-only**.

## (run first) Colab setup

Fetches the data + shared modules from R2. **No-op if you already have the repo locally** (it detects a checkout and skips). On Colab / a bare session it recreates the minimal repo layout under the working directory so the config cell below finds everything unchanged.

In [1]:
# === COLAB BOOTSTRAP - fetch curriculum data + modules from R2 (NO-OP if the repo is local) ===
from pathlib import Path
import sys, urllib.request, urllib.parse, tarfile, subprocess

R2 = 'https://pub-2cee87f70da64080ab70ee0a34b55099.r2.dev/curriculum'
USE_CLEAN = False   # False: raw .xlsx path (JN1's messy-data lesson).  True (skip-ingest): permits_clean.*

_here = Path.cwd()
_have_repo = (_here/'scripts'/'build_v2').exists() or any((p/'scripts'/'build_v2').exists() for p in _here.parents)

def _get(url):
    # r2.dev sits behind Cloudflare, which 403s the default 'Python-urllib' User-Agent; send a browser UA.
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read()

if _have_repo:
    print('local repo detected - no fetch needed')
else:
    try:
        import pyarrow  # the parquet / USE_CLEAN path needs it; Colab has pandas, maybe not pyarrow
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'], check=True)
    def _fetch(url, dest):
        dest = Path(dest)
        if dest.exists():
            return                                   # cached: re-runs don't re-download
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(_get(url)); print('fetched', dest.name)
    # 1) shared modules -> ./scripts/...  (the config-cell repo-root walk then finds scripts/build_v2)
    if not (_here/'scripts'/'build_v2').exists():
        Path('modules.tgz').write_bytes(_get(f'{R2}/curriculum_modules.tar.gz'))
        _tar = tarfile.open('modules.tgz')
        try: _tar.extractall(_here, filter='data')      # py3.12+: safe extract, no deprecation warning
        except TypeError: _tar.extractall(_here)         # older python has no filter arg
        _tar.close(); Path('modules.tgz').unlink(missing_ok=True)   # tidy: drop the intermediate tarball
        print('extracted modules -> ./scripts/')
    # 2) data -> the SAME relative paths the notebooks use (raw .xlsx AND clean exports, both fetched)
    for rel in ['data/raw/cpra-downloads/BP_Annual Permit Report-2018-2022.xlsx',
                'data/raw/cpra-downloads/BP_Annual Permit Report-2023-2025.xlsx',
                'databases/hcd_apr_mirror_2026-06-17_fresh.db',
                'databases/hcd_apr_mirror.db',
                'data/processed/permits_clean.csv',
                'data/processed/permits_clean.parquet',
                'data/processed/permits_clean_README.md']:
        _fetch(f"{R2}/data/{urllib.parse.quote(rel.split('/')[-1])}", _here/rel)   # quote -> %20 for the spaced .xlsx names
    print('curriculum bundle ready (fetched from R2)')


local repo detected - no fetch needed


## Config

In [2]:
# === CONFIG - point this at YOUR city's permit data (this notebook is clonable) ===
from pathlib import Path
import sys, glob
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'scripts' / 'build_v2').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
PERMIT_GLOB = str(REPO_ROOT / 'data/raw/cpra-downloads/BP_Annual Permit Report-*.xlsx')
HEADER_ROW  = 7
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
sys.path.insert(0, str(REPO_ROOT / 'scripts' / 'build_v2'))
print('repo root:', REPO_ROOT)


repo root: /Users/johngage/berkeley-data


## Rebuild the JN3 spine (recap)

In [3]:
import pandas as pd
from collections import defaultdict
from housing_predicates import is_housing, net_units   # JN3's predicates
from s0_keys import normalize_address                   # JN2's key
from cpra_dedup import extract_master_permit            # a '-REV'/'-DEF' is a revision, not a new permit

def load(path):
    d = pd.read_excel(path, dtype=str, header=HEADER_ROW); d.columns = [str(c).strip() for c in d.columns]; return d
df = pd.concat([load(f) for f in glob.glob(PERMIT_GLOB)], ignore_index=True)
df = df[df['PermitNumber'].notna()].copy()
df = df.rename(columns={'Issuance Date': 'IssuanceDate', 'Finaled Date': 'FinaledDate'})
df['isnew'] = df['Work Type'].astype(str).str.strip() == 'New'
df = df[[is_housing(o, u, n, a) for o, u, n, a in zip(df['OccType'], df['UnitsAdded'], df['NumberUnits'], df['ADU'])]]
print(f'{len(df):,} housing permit rows')


31,314 housing permit rows


## A small trap: mixed date formats

Before trusting a date column, look at it. This feed stores **Issuance** dates as `MM/DD/YYYY` and **Finaled** dates as ISO datetimes - in the *same* file. A naive `str[:10]` parser keeps one and silently drops the other (it cost this notebook its 'permitted' buildings on the first pass). Parse robustly with `pd.to_datetime`.

In [4]:
# A SMALL TRAP first: the feed mixes date formats - Issuance is 'MM/DD/YYYY', Finaled is ISO
# datetime. A naive str-slice parser silently drops one of them. Parse robustly.
def pdate(x):
    d = pd.to_datetime(str(x), errors='coerce')
    return d.strftime('%Y-%m-%d') if pd.notna(d) else None
print('issuance  09/10/2020 ->', pdate('09/10/2020'))      # MM/DD/YYYY
print('finaled   2022-01-14 00:00:00 ->', pdate('2022-01-14 00:00:00'))  # ISO datetime


issuance  09/10/2020 -> 2020-09-10
finaled   2022-01-14 00:00:00 -> 2022-01-14


## Milestone selection - the real subtlety

For each building we pick **one** date per milestone from its **master** permits (a `-REV`/`-DEF` sub-permit is a revision, not a new milestone):

- **BP issued = MIN** over the master permits. The *first* permit is when the building started; a later revision must not reset the clock (this is what the RHNA-credit cycle is earned on).
- **CO = the `Finaled` date** (MAX over masters - the building is done when its last real permit finals). Crucially this comes from the structured **`Finaled Status`/`Finaled Date`** columns - a real signal the building is occupiable - **not** parsed from prose. So the event is honest: `is_inferred = 0`.

In [5]:
# Collect, per building, the dates from its MASTER housing-creating permits (REV/DEF excluded).
bld = defaultdict(lambda: {'units': 0.0, 'hasnew': False, 'issue': [], 'final': []})
for r in df.itertuples(index=False):
    st = r.StreetType; st = '' if (st is None or str(st).strip().lower() == 'nan') else str(st)
    k = normalize_address(f'{r.StreetNumber} {r.StreetName} {st}'.strip())
    if not k.number: continue
    b = bld[(k.number, k.street, k.stype)]
    b['units'] = max(b['units'], net_units(r.isnew, r.UnitsAdded, r.NumberUnits, r.ADU))
    if r.isnew: b['hasnew'] = True
    pn = str(r.PermitNumber)
    if extract_master_permit(pn) == pn and net_units(r.isnew, r.UnitsAdded, r.NumberUnits, r.ADU) > 0:
        i, f = pdate(r.IssuanceDate), pdate(r.FinaledDate)
        if i: b['issue'].append((i, pn))
        if f: b['final'].append((f, pn))
spine = {k: b for k, b in bld.items() if b['hasnew'] or b['units'] > 0}

# emit one dated event per (building, milestone):
#   building_permit_issued = MIN over the building's master permits (the FIRST permit starts the clock)
#   co_issued              = MAX finaled over them (the building is done when its last real permit finals)
events = []
for k, b in spine.items():
    if b['issue']: events.append((k, 'building_permit_issued', min(b['issue'])[0]))
    if b['final']: events.append((k, 'co_issued',              max(b['final'])[0]))
n_bp = sum(1 for e in events if e[1] == 'building_permit_issued')
n_co = sum(1 for e in events if e[1] == 'co_issued')
print(f'{len(spine)} buildings -> {n_bp} BP-issued events + {n_co} co_issued events')


1385 buildings -> 1285 BP-issued events + 951 co_issued events


## THE CORE LESSON: derive the stage, don't assert it

A building's **stage** is a *conclusion* from its dated events - **completed** if it has a CO/finaled date, **permitted** if it has only a BP, **pipeline** if neither. We never read a `status='Completed'` string and believe it. (The migration did the opposite - it set stage from a v1 status string, which is how 14 buildings ended up marked 'completed' with no event to back it.)

In [6]:
# STAGE is DERIVED from the dated events - never asserted from a status string.
def stage_of(b):
    if b['final']: return 'completed'    # has a real CO/finaled date
    if b['issue']: return 'permitted'    # a BP issued, but not yet finaled
    return 'pipeline'                    # neither - entitled/in-progress only
from collections import Counter
dist = Counter(stage_of(b) for b in spine.values())
print('stage distribution:', dict(dist), '  (the pipeline S3 = completed 951 / permitted 340 / pipeline 94)')


stage distribution: {'completed': 951, 'pipeline': 94, 'permitted': 340}   (the pipeline S3 = completed 951 / permitted 340 / pipeline 94)


### Every completion traces to a real date

Because each `co_issued` event is the structured `Finaled` date (`is_inferred=0`), **there are zero asserted/guessed completion years** - every completed building can point at the date that proves it. Below: one worked example of each stage.

In [7]:
def show(name, addr):
    k = normalize_address(addr); b = spine[(k.number, k.street, k.stype)]
    bp = min(b['issue'])[0] if b['issue'] else None
    co = max(b['final'])[0] if b['final'] else None
    print(f'  {name:28} units={int(b["units"]):>4}  BP={bp}  CO={co}  -> {stage_of(b)}')
show('2001 Fourth (completed)', '2001 Fourth St')
show('1598 University (permitted)', '1598 University Ave')
show('2711 Shattuck (pipeline)', '2711 Shattuck Ave')

co_dates = [max(b['final'])[0] for b in spine.values() if b['final']]
print(f'\n  {len(co_dates)} completions, all with a structured CO date (0 inferred):', all(co_dates))

  2001 Fourth (completed)      units= 152  BP=2017-05-02  CO=2018-07-31  -> completed
  1598 University (permitted)  units= 207  BP=2024-12-16  CO=None  -> permitted
  2711 Shattuck (pipeline)     units=  22  BP=None  CO=None  -> pipeline

  951 completions, all with a structured CO date (0 inferred): True


## (warning) Honesty note - the 2352 Shattuck thread, now in the DATE dimension

JN3 collapsed Logan Park to **one** 135-unit building. That collapse now bites the *date*: the building's CO is the **MAX finaled = 2023-08-08** (the South building's), which **mis-dates the North's 135 units** - they were finaled **2022-01-14**. Same collapse, new symptom. Still not fixed here; you will resolve it in JN6.

In [8]:
k = normalize_address('2352 Shattuck Ave'); b = spine[(k.number, k.street, k.stype)]
print(f'2352 Shattuck: units={b["units"]:.0f}  CO (MAX-finaled) = {max(b["final"])[0]}')
print('  -> 135 North units carry the South building 2023 date. True North CO = 2022-01-14.')
print('  -> the address-collapse, surfacing in the date dimension. Rediscovered/resolved in JN6.')

2352 Shattuck: units=135  CO (MAX-finaled) = 2023-08-08
  -> 135 North units carry the South building 2023 date. True North CO = 2022-01-14.
  -> the address-collapse, surfacing in the date dimension. Rediscovered/resolved in JN6.


## Checkpoint

In [9]:
# 1) completions count matches the pipeline (S3 = 951)
completed = [k for k, b in spine.items() if b['final']]
assert len(completed) == 951, f'completed {len(completed)} != 951'
# 2) every completion has a real CO date (0 inferred / 0 guessed)
assert all(max(spine[k]['final'])[0] for k in completed)
# 3) one worked example of each stage, derived from events
def st(addr):
    k = normalize_address(addr); return stage_of(spine[(k.number, k.street, k.stype)])
assert st('2001 Fourth St')    == 'completed'
assert st('1598 University Ave') == 'permitted'
assert st('2711 Shattuck Ave')  == 'pipeline'

print('CHECKPOINT PASS')
print(f'  {len(completed)} completions (== S1/S3), all with a structured CO date (0 inferred)')
print('  stage DERIVED from events: 2001 Fourth=completed - 1598 University=permitted - 2711 Shattuck=pipeline')

CHECKPOINT PASS
  951 completions (== S1/S3), all with a structured CO date (0 inferred)
  stage DERIVED from events: 2001 Fourth=completed - 1598 University=permitted - 2711 Shattuck=pipeline


**JN4 done.** Every building now has dated milestones and a stage *derived* from them, with zero asserted completions. **Next - JN5:** tagging each completion with its reporting year and RHNA cycle (and the three date-concepts that must not be conflated).